[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CONNECTS-SCV/bio-guides/blob/main/boltzgen/advanced/03_install_access/03_setup_check.ipynb)


# 03 — 환경 설치·검증

> 본문 [`03_install_access.md`](03_install_access.md) 와 **한 절씩 짝지어** 보세요.
> GPU 진단 셀은 NVIDIA GPU 에서만 의미가 있어요 — Colab 이면 **런타임 → T4 GPU**. 없어도 나머지는 그대로 진행됩니다.

## 0) 준비 — 저장소 클론 & 작업 폴더 이동

이 셀이 저장소를 클론하고 `03_install_access/` 로 이동합니다. 로컬에서 `03_install_access/` 안에 열었다면 클론 없이 진행돼요.

In [ ]:
REPO_URL = "https://github.com/CONNECTS-SCV/bio-guides.git"   # fork 했다면 본인 주소로 바꾸세요
CLONE_AS = "bio-guides"
CHAPTER  = "03_install_access"
PIP_PKGS = ""   # 없으면 설치할 분석 라이브러리

import os, sys, subprocess, pathlib
IN_COLAB = "google.colab" in sys.modules

# HF 가중치 다운로드가 멈춘 채 매달리지 않도록 타임아웃을 겁니다.
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "30")   # 스트림 30초 무응답 → 끊고 재시도
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "15")

def _run(cmd, quiet=False):
    """quiet=True 면 출력을 삼키고 **실패했을 때만** 보여줘요.
    apt-get 은 "(Reading database ... 5%(Reading database ... 10%" 같은 진행률을 600자 넘게 쏟아내는데,
    그게 노트북을 연 학습자가 보는 첫 화면을 덮어버려서 실패로 오해하게 만들거든요."""
    print("$", cmd)
    if not quiet:
        subprocess.run(cmd, shell=True, check=True); return
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.returncode != 0:
        print((p.stdout or "") + (p.stderr or ""))
        raise subprocess.CalledProcessError(p.returncode, cmd)

_MARK = "boltzgen_viz.py"          # 이 파일이 있는 폴더가 advanced/ 루트

def _find_root():
    """advanced/ 루트를 찾습니다."""
    cwd = pathlib.Path.cwd()
    for base in (cwd, *list(cwd.parents)[:3]):
        if (base / _MARK).exists():
            return base
    for pat in (f"*/{_MARK}", f"*/*/{_MARK}", f"*/*/*/{_MARK}"):   # 클론 직후: cwd 아래로만 깊이 3까지
        hits = sorted(cwd.glob(pat))
        if hits:
            return hits[0].parent
    return None

ROOT = _find_root()
if ROOT is None and IN_COLAB:
    if not pathlib.Path(CLONE_AS).exists():
        _run(f'git clone --depth 1 "{REPO_URL}" {CLONE_AS}')
    ROOT = _find_root()
assert ROOT is not None, "advanced/ 루트를 못 찾았습니다. 로컬이면 이 노트북을 챕터 폴더 안에서 여세요."

ADV_ROOT = ROOT.resolve()
os.chdir(ADV_ROOT / CHAPTER)          # 이 챕터 폴더로 이동 → data/ 상대경로가 그대로 동작
sys.path.insert(0, str(ADV_ROOT))     # boltzgen_viz import 보장
import glob as _glob
if IN_COLAB and not _glob.glob("/usr/share/fonts/**/*Nanum*", recursive=True):
    # Colab 에는 한글 폰트가 없어 그래프의 한글 라벨이 □ 로 깨집니다.
    _run("apt-get -qq update", quiet=True)
    _run("DEBIAN_FRONTEND=noninteractive apt-get -qq install -y fonts-nanum", quiet=True)

# import 안 되는 패키지만 설치합니다.
import importlib, importlib.util
_IMPORT_NAME = {"scikit-learn": "sklearn", "pillow": "PIL", "biopython": "Bio"}
def _have(mod):
    try: return importlib.util.find_spec(mod) is not None
    except Exception: return False
def _ensure(pkgs):
    miss = [p for p in pkgs.split() if not _have(_IMPORT_NAME.get(p, p))]
    if miss:
        print("필요 라이브러리 설치:", " ".join(miss))
        _run(f'"{sys.executable}" -m pip -q install ' + " ".join(miss))  # python -m pip (Ch.03 권고)
        importlib.invalidate_caches()
if PIP_PKGS:
    _ensure(PIP_PKGS)

# 내 결과는 my_run/ 에 쌓이고, 없으면 커밋된 레퍼런스로 폴백합니다.
MY  = pathlib.Path("my_run")
MY.mkdir(exist_ok=True)

def find_one(pattern, ref, quiet=False):
    """my_run/ 에서 먼저 찾고, 없으면 레퍼런스 폴더에서 찾습니다."""
    for base in (MY/"final_ranked_designs", MY/"intermediate_designs_inverse_folded", MY):
        hits = sorted(base.glob(pattern))
        if hits:
            if not quiet: print(f"[내 결과]   {hits[0]}")
            return hits[0]
    hits = sorted(pathlib.Path(ref).glob(pattern))
    assert hits, f"{ref} 에서 '{pattern}' 을 찾지 못했습니다."
    if not quiet: print(f"[레퍼런스] {hits[0]}")
    return hits[0]

def cols_in(df, *names):
    """내 실행 결과와 레퍼런스는 컬럼 구성이 조금 다를 수 있어, 있는 컬럼만 고른다."""
    missing = [c for c in names if c not in df.columns]
    if missing:
        print("(이 실행에는 없는 컬럼 — 건너뜁니다:", ", ".join(missing) + ")")
    return [c for c in names if c in df.columns]

def inherit_run(*rel_paths):
    """앞 챕터에서 돌린 설계 결과를 이어받습니다(없으면 레퍼런스로 폴백)."""
    global MY
    if (MY / "final_ranked_designs").exists():
        return MY
    for rel in rel_paths:
        p = pathlib.Path(rel)
        if (p / "final_ranked_designs").exists():
            print("[이어받기] 앞 챕터에서 직접 돌린 결과를 사용합니다:", p)
            MY = p
            return MY
    return MY

# 레퍼런스 그림을 덮어쓰지 않도록 my_ 접두어.
def my_fig(name):
    return f"my_{name}"

print("작업 폴더 :", pathlib.Path.cwd())

## 1) BoltzGen 설치 (본문 3.2)

설치 자체는 한 줄이에요. 진짜 관문은 그다음 CUDA 정합이고, 2)부터 그걸 한 층씩 확인합니다.
가중치(~6GB)는 아직 받지 않고, 첫 `boltzgen run` 때 자동으로 내려받아요.

In [ ]:
import sys
if IN_COLAB:
    _run(f'"{sys.executable}" -m pip -q install boltzgen')
    # cuequivariance 커널은 cuBLAS >= 12.5 를 요구(본문 3.3) — Colab 이미지에 따라 정합을 보강합니다.
    _run(f'"{sys.executable}" -m pip -q install "nvidia-cublas-cu12>=12.5" || true')
else:
    print("로컬은 conda 로 만든 boltzgen_env 를 활성화한 뒤 이 노트북을 여세요 (본문 3.2).")

## 2) 1층 — 드라이버가 지원하는 CUDA 상한 (본문 3.3)

맞춰야 할 층은 넷이에요. **드라이버 ↔ PyTorch ↔ cuequivariance ↔ cuBLAS**.
맨 아래부터 봅니다. `nvidia-smi` 우상단 `CUDA Version: 12.x` 가 드라이버가 지원하는 **상한**이고,
위층의 CUDA 는 이 값을 넘으면 안 돼요.

In [ ]:
import shutil, subprocess
if shutil.which("nvidia-smi"):
    out = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
    for line in out.splitlines()[:12]:
        print(line)
else:
    print("nvidia-smi 가 없어요 — 지금은 CPU 런타임입니다.")
    print("  · 아래 GPU 항목은 비어서 나오고, 설치·CLI·명세 검증은 그대로 진행됩니다.")

## 3) 2층 — PyTorch 의 CUDA 빌드와 GPU 인식 (본문 3.3)

`torch.version.cuda` 가 2)의 상한 이하 12.x 이고 `cuda available` 이 True 면 통과예요.
major 가 어긋나면(예: 드라이버 12.4 에 torch cu130) `driver too old` 로 GPU 를 아예 못 봅니다.
여기서 읽은 GPU 이름·capability·VRAM 은 뒤 셀에서도 그대로 씁니다.

In [ ]:
CAP, VRAM = None, None
try:
    import torch
    print("torch", torch.__version__, "| built cuda", torch.version.cuda)
    print("cuda available", torch.cuda.is_available())
    if torch.cuda.is_available():
        CAP = torch.cuda.get_device_capability()
        VRAM = torch.cuda.get_device_properties(0).total_memory / 1024**3
        a = torch.randn(256, 256, device="cuda"); b = torch.randn(256, 256, device="cuda")
        print(f"device {torch.cuda.get_device_name(0)} | capability {CAP} | VRAM {VRAM:.1f} GB")
        print("matmul 검산", bool(torch.isfinite((a @ b).sum())))
except Exception as e:
    print("torch import/실행 실패 —", repr(e)[:160])
    print("  → Colab 이면 런타임 재시작 후 재실행, 로컬이면 드라이버 상한에 맞는 빌드로 재설치")
    print("     python -m pip install 'torch==2.6.0' --index-url https://download.pytorch.org/whl/cu124"
          " --extra-index-url https://pypi.org/simple")

## 4) 3층 — cuequivariance 가속 커널 (본문 3.3)

torch 가 GPU 를 봐도 여기서 한 번 더 막힐 수 있어요. import 가 되면 통과입니다.

> capability 8 미만 GPU(T4 등)는 이 커널을 **애초에 쓰지 않아** 실패해도 설계는 그대로 돌아가요.

In [ ]:
try:
    from cuequivariance_ops_torch import triangle_multiplicative_update
    print("cuequivariance kernel OK")
except Exception as e:
    print("FAILED —", repr(e)[:160])
    print("  → undefined symbol: cublasGemmGroupedBatchedEx 라면 cuBLAS 문제예요. 5) 셀에서 버전을 봅니다.")

## 5) 4층 — cuBLAS 버전 (본문 3.3)

4)의 커널이 요구하는 함수 `cublasGemmGroupedBatchedEx` 는 **cuBLAS 12.5 이상**에서 추가됐어요.
torch 가 12.4 를 번들로 들고 오면 커널 로딩만 실패합니다. cuBLAS 는 12.x 안에서 하위 호환이라
이것만 올려도 torch 는 그대로 돌아가요.

In [ ]:
import subprocess, sys
out = subprocess.run([sys.executable, "-m", "pip", "show", "nvidia-cublas-cu12"],
                     capture_output=True, text=True).stdout
ver = next((l.split(":", 1)[1].strip() for l in out.splitlines() if l.startswith("Version:")), None)
print("nvidia-cublas-cu12", ver or "(설치 안 됨)")

if ver and tuple(int(x) for x in ver.split(".")[:2]) >= (12, 5):
    print("판정 통과 — 12.5 이상")
elif ver:
    print("판정 미달 — 12.5 미만이라 4)의 커널이 실패해요.")
    print("  → python -m pip install 'nvidia-cublas-cu12==12.9.2.10' 후 런타임 재시작")
else:
    print("판정 보류 — GPU 로 설계할 계획이면 미리 넣어두세요.")
    print("  → python -m pip install 'nvidia-cublas-cu12>=12.5'")

## 6) 실행 로그를 미리 읽어두기 — 커널과 배치 (본문 3.1.5)

네 층이 맞았으면 이제 `boltzgen run` 로그예요. 딱 두 줄만 알면 실행을 통제할 수 있어요.

```
Using kernels: False [device capability: (7, 5)]
Using diffusion batch size: 1
```

첫 줄은 4)에서 본 가속 커널이 켜졌는지, 둘째 줄은 유일하게 신경 쓸 메모리 레버예요.
`--diffusion_batch_size` 를 지정하지 않으면 `num_designs` 가 **100 미만이면 1, 100 이상이면 10** 으로
자동 결정됩니다. 즉 99 → 100 으로 늘리는 순간이 분기점이에요.

> 가장 무거운 folding 단계는 내부적으로 `batch_size: 1` 고정이라, 디자인을 몇 개 뽑든
> GPU 에 한 번에 올라가는 복합체는 하나예요.

In [ ]:
import pandas as pd
from IPython.display import display

print("--diffusion_batch_size 를 안 주면 num_designs 만 보고 자동으로 정해져요 (100 이 분기점)")
display(pd.DataFrame([
    {"--num_designs": n, "diffusion 배치 자동값": 1 if n < 100 else 10,
     "비고": "이 가이드 실습 규모" if n <= 30 else ("분기점 바로 아래" if n == 99 else
             ("분기점 — 여기서 10 으로 뜀" if n == 100 else "프로덕션 규모"))}
    for n in (4, 30, 99, 100, 1000)]))
print()
if CAP is None:
    print("GPU 가 없어 지금은 표만 읽고 갑니다 — 실습 예제는 전부 num_designs 4~30 이라 배치는 1이에요.")
else:
    print("가속 커널", "ON" if CAP[0] >= 8 else "OFF (capability 8 미만 — 자동 비활성, 정상 동작하고 조금 느림)")
    print(f"이 GPU 의 VRAM {VRAM:.1f} GB — 실습 규모(num_designs 4~30)는 배치가 1이라 옵션이 필요 없어요.")
    print("100개 이상 뽑을 때 메모리가 빠듯하면 --diffusion_batch_size 1 을 명시하세요.")
    print("CUDA out of memory 가 뜨면 배치를 줄이고, 그래도 안 되면 num_designs 를 쪼개 돌린 뒤 boltzgen merge 로 합칩니다.")
    if CAP[0] < 8:
        print("bf16 네이티브 미지원 카드(T4/V100)에서 정밀도 오류가 나면 --config folding trainer.precision=32 (부록 A6)")

## 7) 모델 가중치와 캐시 (본문 3.4)

가중치는 첫 실행 때 HuggingFace Hub 에서 자동으로 받아요. 약 6GB 이고 한 번 받으면
`~/.cache/huggingface/` 에 남아 재사용됩니다. 지금 무엇이 받아져 있는지부터 봅니다.

In [ ]:
import os, pathlib
cache = pathlib.Path(os.environ.get("HF_HOME", pathlib.Path.home() / ".cache" / "huggingface"))
print("캐시 경로", cache, "| 존재", cache.exists())
if cache.exists():
    hits = [p for p in cache.rglob("*")                       # snapshots/ 만 봐야 blob 중복이 안 잡혀요
            if p.is_file() and "snapshots" in p.parts and "boltz" in str(p).lower()]
    used = sum(p.stat().st_size for p in hits) / 1024**3
    print(f"BoltzGen 관련 캐시 파일 {len(hits)}개 | {used:.2f} GB")
    for p in sorted(hits, key=lambda x: -x.stat().st_size)[:6]:
        print(f"    {p.name:34s}{p.stat().st_size / 1024**2:8.1f} MB")
print()
print("미리 받아두려면 boltzgen download")
print("  --cache <DIR>       모델 캐시 위치 지정 (기본 ~/.cache)")
print("  --models_token <T>  HuggingFace 토큰 (rate limit 완화·빠른 다운로드)")
print("  --force_download    재다운로드")
print()
print("전부 받으면 백본 2종·역접힘·구조검증 체크포인트에 보조 데이터까지 약 6GB 예요.")
print("지금 비어 있어도 정상이고, 다운로드가 느리거나 실패하면 --models_token 이나 --force_download 를 쓰세요.")

## 8) CLI 확인 (본문 3.6)

여기부터는 설치 자체의 마무리 점검이에요. 버전과 서브커맨드가 뜨면 실행 파일이 제대로 깔린 겁니다.

In [ ]:
import shutil, subprocess
if shutil.which("boltzgen"):
    print(subprocess.run(["boltzgen", "--version"], capture_output=True, text=True).stdout.strip())
    helptext = subprocess.run(["boltzgen", "--help"], capture_output=True, text=True).stdout
    for line in helptext.splitlines()[:14]:
        print(line)
    print()
    print("run / configure / execute / download / check / merge 가 보이면 정상이에요.")
else:
    print("boltzgen 이 아직 없어요 — 1) 셀을 먼저 실행하세요.")

## 9) 설계 명세 검증 (본문 3.6)

마지막은 가벼운 엔드투엔드 확인이에요. `boltzgen check` 는 명세를 읽고 타깃 구조까지 파싱하므로,
여기까지 통과하면 설치·의존성·데이터 경로가 모두 살아 있다는 뜻입니다. GPU 는 필요 없어요.

In [ ]:
import shutil, subprocess, pathlib
SRC = ADV_ROOT / ".boltzgen_src"          # Ch.02 에서 이미 받았다면 그대로 재사용합니다
if shutil.which("boltzgen") and not SRC.exists():
    _run(f'git clone --depth 1 https://github.com/HannesStark/boltzgen.git "{SRC}"')

spec = SRC / "example/vanilla_protein/1g13prot.yaml"
if shutil.which("boltzgen") and spec.exists():
    r = subprocess.run(["boltzgen", "check", str(spec)], capture_output=True, text=True)
    print(r.stdout.strip() if r.returncode == 0 else r.stderr.strip()[-700:])
    print()
    print("이 예제는 sequence 가 80..140 이라 Total designed residues 가 그 범위 안이면 통과예요.")
    print("매 실행마다 값이 달라지는 것도 정상입니다.")
else:
    print("boltzgen 또는 예제 명세가 아직 없어요 — 1) 셀을 실행한 뒤 다시 오세요.")

### 정리

드라이버 → torch → 커널 → cuBLAS 네 층을 순서대로 통과시키고, 가중치 캐시와 CLI, 명세 검증까지 확인했어요.
설치에서 막히는 대부분은 이 순서로 짚으면 어느 층이 어긋났는지 바로 드러납니다.
이제 실제로 6스텝을 끝까지 돌릴 차례예요 — Ch.04 에서 `num_designs 4`, `budget 2` 규모로 파이프라인을 완주합니다.